# CIFAR-10: lean, controlled SLA+SD notebook for NYU HPC

This is a compact competition workflow: it reads the permitted course images from `/data`, compares
predeclared candidates on a fixed stratified 45k/5k split, retrains the selected single model on all
50k labelled images, then writes `submission.csv`. Evaluation images are used only for final inference.

The default compares 200 and 300 training epochs. Temperature-scaled self-distillation, SWA, SLA-CP,
AugMix/JSD, and SE blocks are separate opt-in candidates. They are intentionally not combined unless a
controlled holdout experiment establishes that combination. No external data, pretrained weights,
teacher model, or multi-model submission ensemble is used.

In [1]:
import copy, json, math, os, random, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.optim.swa_utils import AveragedModel
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

def is_data_root(path):
    required = ("train_labels.csv", "test.csv", "sample_submission.csv", "train_images", "test_images")
    return path.is_dir() and all((path / item).exists() for item in required)

configured_data_dir = os.environ.get("CIFAR10_DATA_DIR")
data_candidates = [Path(configured_data_dir).expanduser()] if configured_data_dir else []
data_candidates += [Path("/data"), Path.home() / "data", Path.cwd() / "data"]
DATA_DIR = next((path for path in data_candidates if is_data_root(path)), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find the competition data. Set CIFAR10_DATA_DIR or place it in /data or ~/data.")
WORK_DIR = Path(os.environ.get("CIFAR10_WORK_DIR", str(Path(os.environ.get("SCRATCH", str(Path.home()))) / "cifar10_lean_sla")))
SUBMISSION_PATH = Path(os.environ.get("CIFAR10_SUBMISSION_PATH", str(Path.home() / "submission.csv"))).expanduser()
CACHE_PATH, RUNS_DIR = WORK_DIR / "arrays.npz", WORK_DIR / "runs"
for directory in (WORK_DIR, RUNS_DIR, SUBMISSION_PATH.parent): directory.mkdir(parents=True, exist_ok=True)

SEEDS = (123,)                 # Use (123, 456, 789) to measure seed variation; no ensemble is submitted.
FINAL_SEED = SEEDS[0]
VAL_PER_CLASS = 500
BATCH_SIZE = 256               # 4 rotations => 1,024 images through the backbone per SLA step.
EVAL_BATCH_SIZE = 128
NUM_WORKERS = min(8, max(2, int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 4)) - 1))
WARMUP_EPOCHS, WEIGHT_DECAY, LABEL_SMOOTHING = 5, 5e-4, 0.1
TTA_MODE = "tta6"             # Fixed before training: identity, flip, and two symmetric one-pixel shifts.
CHECKPOINT_EVERY, FORCE_RETRAIN = 10, False

# Only the matched 200-vs-300 schedule comparison runs by default.
RUN_BASELINE_200, RUN_LONGER_300 = True, True
RUN_TEMPERATURE_2, RUN_SWA_300 = False, False
RUN_SLA_CP, RUN_AUGMIX_JSD, RUN_SE = False, False, False
RUN_FINAL_RETRAIN = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda": raise RuntimeError("Start an OOD JupyterLab session with a GPU allocation.")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")
print("GPU:", torch.cuda.get_device_name(0), "| data:", DATA_DIR, "| work:", WORK_DIR)

GPU: NVIDIA A100-SXM4-40GB | data: /home/bap9622/data | work: /scratch/bap9622/cifar10_lean_sla


In [2]:
required = ("train_labels.csv", "test.csv", "sample_submission.csv", "train_images", "test_images")
for item in required: assert (DATA_DIR / item).exists(), f"Missing {DATA_DIR / item}"
train_df, test_df = pd.read_csv(DATA_DIR / "train_labels.csv"), pd.read_csv(DATA_DIR / "test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
assert {"id", "filename", "label"}.issubset(train_df.columns)
assert {"id", "filename"}.issubset(test_df.columns) and sample.columns.tolist() == ["id", "label"]
assert np.array_equal(test_df.id.to_numpy(), sample.id.to_numpy())

def read_rgb(filename, folder):
    with Image.open(DATA_DIR / folder / str(filename)) as image:
        array = np.asarray(image.convert("RGB"), dtype=np.uint8)
    if array.shape != (32, 32, 3): raise ValueError(f"Expected 32x32 RGB: {filename}")
    return array

if CACHE_PATH.exists():
    cache = np.load(CACHE_PATH)
    x_train, y_train, x_test = cache["x_train"], cache["y_train"], cache["x_test"]
else:
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
        x_train = np.stack(list(pool.map(lambda f: read_rgb(f, "train_images"), train_df.filename)))
        x_test = np.stack(list(pool.map(lambda f: read_rgb(f, "test_images"), test_df.filename)))
    y_train = train_df.label.to_numpy(np.int64)
    np.savez_compressed(CACHE_PATH, x_train=x_train, y_train=y_train, x_test=x_test)
assert x_train.shape == (50_000, 32, 32, 3) and x_test.shape == (2_000, 32, 32, 3)
assert np.isin(y_train, np.arange(10)).all()

split_rng = np.random.default_rng(1123)
val_idx = np.sort(np.concatenate([split_rng.choice(np.flatnonzero(y_train == label), VAL_PER_CLASS, replace=False) for label in range(10)]))
train_idx = np.setdiff1d(np.arange(len(y_train)), val_idx, assume_unique=True)
assert len(train_idx) == 45_000 and len(val_idx) == 5_000
print("train/validation/test:", len(train_idx), len(val_idx), len(x_test))

train/validation/test: 45000 5000 2000


In [3]:
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)

class Cutout:
    def __init__(self, size=12): self.size = size
    def __call__(self, image):
        _, height, width = image.shape
        top = torch.randint(0, height - self.size + 1, ()).item()
        left = torch.randint(0, width - self.size + 1, ()).item()
        image = image.clone(); image[:, top:top + self.size, left:left + self.size] = 0
        return image

geometry = transforms.Compose([transforms.RandomCrop(32, padding=4, padding_mode="reflect"), transforms.RandomHorizontalFlip()])
to_model = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD), Cutout(12)])
trivial = transforms.TrivialAugmentWide()
augmix = transforms.AugMix(
    severity=3,
    mixture_width=3,
    chain_depth=-1,
    alpha=1.0,
)
evaluation_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class TrainImages(Dataset):
    def __init__(self, images, labels, recipe): self.images, self.labels, self.recipe = images, labels, recipe
    def __len__(self): return len(self.images)
    def __getitem__(self, index):
        base = geometry(Image.fromarray(self.images[index]))
        if self.recipe == "trivial":
            return to_model(trivial(base)), int(self.labels[index])
        clean, one, two = to_model(base), to_model(augmix(base)), to_model(augmix(base))
        return (clean, one, two), int(self.labels[index])

class EvalImages(Dataset):
    def __init__(self, images, labels=None): self.images, self.labels = images, labels
    def __len__(self): return len(self.images)
    def __getitem__(self, index):
        image = evaluation_transform(Image.fromarray(self.images[index]))
        return (image, int(self.labels[index])) if self.labels is not None else image

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def seed_worker(_):
    worker_seed = torch.initial_seed() % 2**32
    random.seed(worker_seed); np.random.seed(worker_seed)

def loader(images, labels, recipe, seed, train):
    dataset = TrainImages(images, labels, recipe) if train else EvalImages(images, labels)
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(dataset, batch_size=BATCH_SIZE if train else EVAL_BATCH_SIZE, shuffle=train,
        drop_last=train, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0,
        worker_init_fn=seed_worker, generator=generator)

In [4]:
class SqueezeExcite(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__(); hidden = max(8, channels // reduction)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc1, self.fc2 = nn.Conv2d(channels, hidden, 1), nn.Conv2d(hidden, channels, 1)
    def forward(self, x):
        gate = torch.sigmoid(self.fc2(F.relu(self.fc1(self.pool(x)))))
        return x * gate

class PreActBlock(nn.Module):
    def __init__(self, c_in, c_out, stride, use_se):
        super().__init__()
        self.bn1, self.conv1 = nn.BatchNorm2d(c_in), nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False)
        self.bn2, self.conv2 = nn.BatchNorm2d(c_out), nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False)
        self.skip = None if stride == 1 and c_in == c_out else nn.Conv2d(c_in, c_out, 1, stride, bias=False)
        self.se = SqueezeExcite(c_out) if use_se else nn.Identity()
    def forward(self, x):
        activated = F.relu(self.bn1(x)); shortcut = x if self.skip is None else self.skip(activated)
        return self.se(self.conv2(F.relu(self.bn2(self.conv1(activated))))) + shortcut

class PreActResNet(nn.Module):
    def __init__(self, auxiliary="sla", use_se=False):
        super().__init__()
        widths, depths, channels = (64, 128, 256), (4, 4, 3), 64
        self.auxiliary = auxiliary
        self.stem = nn.Conv2d(3, channels, 3, 1, 1, bias=False)
        blocks = []
        for stage, (width, depth) in enumerate(zip(widths, depths)):
            for block in range(depth):
                blocks.append(PreActBlock(channels, width, 2 if stage and block == 0 else 1, use_se)); channels = width
        self.blocks, self.bn = nn.Sequential(*blocks), nn.BatchNorm2d(channels)
        self.head = nn.Linear(channels, 10)
        self.joint_head = nn.Linear(channels, 40 if auxiliary == "sla" else 240)
        for module in self.modules():
            if isinstance(module, nn.Conv2d): nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
    def features(self, x):
        return F.adaptive_avg_pool2d(F.relu(self.bn(self.blocks(self.stem(x)))), 1).flatten(1)
    def forward(self, x): return self.head(self.features(x))

def build_model(config):
    model = PreActResNet(config["auxiliary"], config["use_se"]).to(DEVICE).to(memory_format=torch.channels_last)
    count = sum(parameter.numel() for parameter in model.parameters())
    assert count <= 5_000_000, f"{count:,} exceeds the 5M parameter limit"
    print(f"{config['name']}: {count:,} trainable parameters")
    return model

In [5]:
PERMUTATIONS = torch.tensor(((0, 1, 2), (0, 2, 1), (1, 0, 2), (1, 2, 0), (2, 0, 1), (2, 1, 0)), device=DEVICE)

def rotate4(images): return torch.stack([torch.rot90(images, k, (2, 3)) for k in range(4)], 1).flatten(0, 1)

def permute_rgb(images, indices):
    channels = PERMUTATIONS[indices]
    return torch.gather(images, 1, channels[:, :, None, None].expand_as(images))

def aggregate_joint(joint_logits, batch, auxiliary, colour=None):
    transforms_per_class = 4 if auxiliary == "sla" else 24
    joint = joint_logits.float().view(batch, 4, 10, transforms_per_class)
    if auxiliary == "sla": return torch.stack([joint[:, k, :, k] for k in range(4)], 1).mean(1)
    colour = torch.zeros(batch, dtype=torch.long, device=joint.device) if colour is None else colour
    values = []
    for k in range(4):
        index = (k * 6 + colour).view(batch, 1, 1).expand(batch, 10, 1)
        values.append(joint[:, k].gather(2, index).squeeze(2))
    return torch.stack(values, 1).mean(1)

def sla_sd_loss(model, images, labels, config, augmix_views=None):
    batch, auxiliary = len(labels), config["auxiliary"]
    colour = torch.randint(0, 6, (batch,), device=DEVICE) if auxiliary == "sla_cp" else None
    base = permute_rgb(images, colour) if colour is not None else images
    features = model.features(rotate4(base).contiguous(memory_format=torch.channels_last))
    joint_logits, class_logits = model.joint_head(features), model.head(features[::4])
    if auxiliary == "sla":
        joint_targets = (labels[:, None] * 4 + torch.arange(4, device=DEVICE)).flatten()
    else:
        joint_targets = (labels[:, None] * 24 + torch.arange(4, device=DEVICE)[None] * 6 + colour[:, None]).flatten()
    joint_loss = F.cross_entropy(joint_logits.float(), joint_targets, label_smoothing=LABEL_SMOOTHING)
    class_loss = F.cross_entropy(class_logits.float(), labels, label_smoothing=LABEL_SMOOTHING)
    teacher = aggregate_joint(joint_logits, batch, auxiliary, colour).detach()
    temperature = config["temperature"]
    distill = F.kl_div(F.log_softmax(class_logits.float() / temperature, 1), F.softmax(teacher / temperature, 1), reduction="batchmean") * temperature**2
    loss = joint_loss + class_loss + config["sd_weight"] * distill
    if augmix_views is not None:
        one, two = augmix_views
        one_logits, two_logits = model(one).float(), model(two).float()
        mixture = (F.softmax(class_logits.float(), 1) + F.softmax(one_logits, 1) + F.softmax(two_logits, 1)) / 3
        jsd = (F.kl_div(F.log_softmax(class_logits.float(), 1), mixture, reduction="batchmean") +
               F.kl_div(F.log_softmax(one_logits, 1), mixture, reduction="batchmean") +
               F.kl_div(F.log_softmax(two_logits, 1), mixture, reduction="batchmean")) / 3
        loss = loss + config["jsd_weight"] * jsd
    return loss

def make_optimizer(model, config, steps_per_epoch):
    decayed, undecayed = [p for p in model.parameters() if p.ndim > 1], [p for p in model.parameters() if p.ndim <= 1]
    options = dict(lr=0.1 * BATCH_SIZE / 128, momentum=0.9, nesterov=True)
    try: optimizer = torch.optim.SGD(({"params": decayed, "weight_decay": WEIGHT_DECAY}, {"params": undecayed, "weight_decay": 0.0}), fused=True, **options)
    except (RuntimeError, TypeError): optimizer = torch.optim.SGD(({"params": decayed, "weight_decay": WEIGHT_DECAY}, {"params": undecayed, "weight_decay": 0.0}), **options)
    total, warmup = config["epochs"] * steps_per_epoch, WARMUP_EPOCHS * steps_per_epoch
    schedule = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: (step + 1) / warmup if step < warmup else 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, total - warmup))))
    return optimizer, schedule

@torch.inference_mode()
def model_scores(model, images, auxiliary):
    features = model.features(rotate4(images).contiguous(memory_format=torch.channels_last))
    head = model.head(features[::4]).float()
    aggregate = aggregate_joint(model.joint_head(features), len(images), auxiliary)
    return F.softmax(head, 1) + F.softmax(aggregate, 1)

def shifted(images, dy, dx):
    height, width = images.shape[-2:]; padded = F.pad(images, (1, 1, 1, 1), mode="reflect")
    return padded[:, :, 1 + dy:1 + dy + height, 1 + dx:1 + dx + width]

@torch.inference_mode()
def probabilities(model, data_loader, auxiliary):
    model.eval(); output = []
    for batch in data_loader:
        images = batch[0] if isinstance(batch, (tuple, list)) else batch
        images = images.to(DEVICE, non_blocking=True).contiguous(memory_format=torch.channels_last)
        views = [(1.0, images)] if TTA_MODE == "none" else [(0.5, images), (0.5, images.flip(-1))]
        if TTA_MODE == "tta6":
            views = [(0.25, images), (0.25, images.flip(-1)), (0.125, shifted(images, -1, -1)), (0.125, shifted(images, 1, 1)), (0.125, shifted(images.flip(-1), -1, -1)), (0.125, shifted(images.flip(-1), 1, 1))]
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            output.append(sum(weight * model_scores(model, view, auxiliary) for weight, view in views))
    return torch.cat(output)

In [6]:
def refresh_batchnorm(model, train_loader):
    batchnorms = [module for module in model.modules() if isinstance(module, nn.modules.batchnorm._BatchNorm)]
    if not batchnorms: return
    momenta = {module: module.momentum for module in batchnorms}
    for module in batchnorms: module.reset_running_stats(); module.momentum = None
    model.train()
    with torch.no_grad():
        for data in train_loader:
            images = data[0][0] if isinstance(data[0], (tuple, list)) else data[0]
            images = images.to(DEVICE, non_blocking=True).contiguous(memory_format=torch.channels_last)
            model(images)
    for module, momentum in momenta.items(): module.momentum = momentum

def save_checkpoint(path, payload):
    temporary = path.with_suffix(".tmp"); torch.save(payload, temporary); os.replace(temporary, path)

def train_run(name, config, indices, validation_indices, seed):
    run_dir = RUNS_DIR / f"{name}_s{seed}"; run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path, model_path = run_dir / "checkpoint.pt", run_dir / "model.pt"
    seed_everything(seed)
    train_loader = loader(x_train[indices], y_train[indices], config["recipe"], seed, train=True)
    model = build_model(config)
    swa_model = AveragedModel(model).to(DEVICE).to(memory_format=torch.channels_last) if config["swa_start"] else None
    optimizer, scheduler = make_optimizer(model, config, len(train_loader))
    try: scaler = torch.amp.GradScaler("cuda", enabled=True)
    except AttributeError: scaler = torch.cuda.amp.GradScaler(enabled=True)
    start_epoch, swa_updates = 0, 0
    if checkpoint_path.exists() and not FORCE_RETRAIN:
        try: checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
        except TypeError: checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
        if checkpoint["config"] != config: raise RuntimeError(f"Checkpoint configuration differs: {checkpoint_path}")
        model.load_state_dict(checkpoint["model"]); optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"]); scaler.load_state_dict(checkpoint["scaler"])
        if swa_model is not None and checkpoint["swa"] is not None: swa_model.load_state_dict(checkpoint["swa"])
        start_epoch, swa_updates = checkpoint["epoch"], checkpoint["swa_updates"]
        print(f"[{name}] resumed after epoch {start_epoch}")
    for epoch in range(start_epoch, config["epochs"]):
        model.train(); total, count, started = 0.0, 0, time.time()
        for data, labels in train_loader:
            if config["recipe"] == "augmix":
                clean, one, two = (item.to(DEVICE, non_blocking=True).contiguous(memory_format=torch.channels_last) for item in data)
                augmix_views = (one, two)
            else:
                clean, augmix_views = data.to(DEVICE, non_blocking=True).contiguous(memory_format=torch.channels_last), None
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16): loss = sla_sd_loss(model, clean, labels, config, augmix_views)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); scheduler.step()
            total += loss.item() * len(labels); count += len(labels)
        if swa_model is not None and epoch + 1 >= config["swa_start"]:
            swa_model.update_parameters(model); swa_updates += 1
        print(f"[{name}] epoch {epoch + 1:03d}/{config['epochs']} loss={total / count:.4f} lr={optimizer.param_groups[0]['lr']:.5f} time={time.time() - started:.1f}s", flush=True)
        if (epoch + 1) % CHECKPOINT_EVERY == 0 or epoch + 1 == config["epochs"]:
            save_checkpoint(checkpoint_path, {"config": config, "epoch": epoch + 1, "model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(), "swa": None if swa_model is None else swa_model.state_dict(), "swa_updates": swa_updates})
    chosen = swa_model if swa_model is not None and swa_updates else model
    if chosen is swa_model: refresh_batchnorm(chosen, train_loader)
    chosen.eval(); torch.save(chosen.state_dict(), model_path)
    result = {"name": name, "seed": seed, "config": config, "params": sum(p.numel() for p in chosen.parameters())}
    if len(validation_indices):
        validation_loader = loader(x_train[validation_indices], y_train[validation_indices], "trivial", seed, train=False)
        prediction = probabilities(chosen, validation_loader, config["auxiliary"])
        result["val_accuracy"] = (prediction.argmax(1).cpu().numpy() == y_train[validation_indices]).mean().item()
        print(f"[{name}] VAL5K {result['val_accuracy'] * 100:.2f}%")
    return chosen, result

In [7]:
def candidate(name, epochs, *, temperature=1.0, swa_start=0, auxiliary="sla", recipe="trivial", use_se=False, jsd_weight=0.0):
    return {"name": name, "epochs": epochs, "temperature": temperature, "sd_weight": 1.0,
            "swa_start": swa_start, "auxiliary": auxiliary, "recipe": recipe, "use_se": use_se,
            "jsd_weight": jsd_weight}

candidates = []
if RUN_BASELINE_200: candidates.append(candidate("sla_e200", 200))
if RUN_LONGER_300: candidates.append(candidate("sla_e300", 300))
if RUN_TEMPERATURE_2: candidates.append(candidate("sla_e300_t2", 300, temperature=2.0))
if RUN_SWA_300: candidates.append(candidate("sla_e300_swa", 300, swa_start=220))
if RUN_SLA_CP: candidates.append(candidate("sla_cp_e200", 200, auxiliary="sla_cp"))
if RUN_AUGMIX_JSD: candidates.append(candidate("sla_augmix_jsd_e200", 200, recipe="augmix", jsd_weight=12.0))
if RUN_SE: candidates.append(candidate("sla_se_e300", 300, use_se=True))
if not candidates: raise RuntimeError("Enable at least one candidate.")

holdout_results = []
for config in candidates:
    for seed in SEEDS:
        _, result = train_run(config["name"], config, train_idx, val_idx, seed)
        holdout_results.append(result)
summary = pd.DataFrame([{"name": item["name"], "seed": item["seed"], "val_accuracy": item["val_accuracy"]} for item in holdout_results])
display(summary)
mean_accuracy = summary.groupby("name", as_index=True)["val_accuracy"].mean()
selected_name = mean_accuracy.idxmax()
selected_config = next(config for config in candidates if config["name"] == selected_name)
print("Selected by mean VAL5K accuracy:", selected_name, f"({mean_accuracy[selected_name] * 100:.2f}%)")

final_model = None
if RUN_FINAL_RETRAIN:
    all_indices = np.arange(len(y_train))
    final_model, _ = train_run(f"final_all50k_{selected_name}", selected_config, all_indices, np.empty(0, dtype=np.int64), FINAL_SEED)

sla_e200: 4,706,674 trainable parameters
[sla_e200] resumed after epoch 200
[sla_e200] VAL5K 97.74%
sla_e300: 4,706,674 trainable parameters
[sla_e300] resumed after epoch 300
[sla_e300] VAL5K 97.82%


,name,seed,val_accuracy
0,sla_e200,123,0.9774
1,sla_e300,123,0.9782


Selected by mean VAL5K accuracy: sla_e300 (97.82%)
sla_e300: 4,706,674 trainable parameters
[final_all50k_sla_e300] resumed after epoch 180
[final_all50k_sla_e300] epoch 181/300 loss=2.2545 lr=0.07011 time=20.9s
[final_all50k_sla_e300] epoch 182/300 loss=2.2440 lr=0.06910 time=10.2s
[final_all50k_sla_e300] epoch 183/300 loss=2.2532 lr=0.06809 time=10.2s
[final_all50k_sla_e300] epoch 184/300 loss=2.2478 lr=0.06708 time=10.2s
[final_all50k_sla_e300] epoch 185/300 loss=2.2496 lr=0.06608 time=10.2s
[final_all50k_sla_e300] epoch 186/300 loss=2.2414 lr=0.06508 time=10.2s
[final_all50k_sla_e300] epoch 187/300 loss=2.2257 lr=0.06408 time=10.2s
[final_all50k_sla_e300] epoch 188/300 loss=2.2183 lr=0.06309 time=10.2s
[final_all50k_sla_e300] epoch 189/300 loss=2.2158 lr=0.06210 time=10.2s
[final_all50k_sla_e300] epoch 190/300 loss=2.2115 lr=0.06112 time=10.2s
[final_all50k_sla_e300] epoch 191/300 loss=2.2043 lr=0.06014 time=10.2s
[final_all50k_sla_e300] epoch 192/300 loss=2.2133 lr=0.05916 time=10

In [8]:
if final_model is None:
    raise RuntimeError("No final all-50k model. Keep RUN_FINAL_RETRAIN=True.")

test_loader = loader(x_test, None, "trivial", FINAL_SEED, train=False)
test_probabilities = probabilities(final_model, test_loader, selected_config["auxiliary"])

submission = pd.DataFrame({
    "id": sample.id.to_numpy(),
    "label": test_probabilities.argmax(1).cpu().numpy().astype(int),
})

assert submission.columns.tolist() == ["id", "label"]
assert len(submission) == len(sample) == 2_000
assert np.array_equal(submission.id.to_numpy(), sample.id.to_numpy())
assert submission.label.between(0, 9).all()

SUBMISSION_PATH = Path.home() / "submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)

print("Wrote:", SUBMISSION_PATH)
display(submission.head())

Wrote: /home/bap9622/submission.csv


,id,label
0,0,6
1,1,3
2,2,3
3,3,1
4,4,3
